In [1]:
!pip install pyspark==3.5.0 delta-spark==3.1.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("DeltaAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [19]:
df = spark.read.csv(
    "customer_master.csv",
    header=True,
    inferSchema=True
)

df.show()

+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|New York|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
+-----------+-------------+-----------------+--------+--------+



In [20]:
print("Total Rows:", df.count())

Total Rows: 4


In [21]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----------+----+-----+-----+--------+
|customer_id|name|email|phone|location|
+-----------+----+-----+-----+--------+
|          0|   0|    0|    0|       0|
+-----------+----+-----+-----+--------+



In [22]:
df = df.na.drop()

In [23]:
df = df.dropDuplicates()

In [24]:
print("Rows after cleaning:", df.count())

Rows after cleaning: 3


In [25]:
df.show()

+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|
|        101|  Alice Smith|  alice@email.com|555-0101|New York|
+-----------+-------------+-----------------+--------+--------+



In [26]:
path = "/content/customer_delta"

In [27]:
df.write.format("delta") \
.mode("overwrite") \
.save(path)

In [28]:
delta_df = spark.read.format("delta").load(path)

delta_df.show()

+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|
|        101|  Alice Smith|  alice@email.com|555-0101|New York|
+-----------+-------------+-----------------+--------+--------+



In [29]:
inc_df = spark.read.csv(
    "customer_incremental.csv",
    header=True,
    inferSchema=True
)

inc_df.show()

+-----------+-----------+---------------+--------+--------+
|customer_id|       name|          email|   phone|location|
+-----------+-----------+---------------+--------+--------+
|        101|Alice Smith|alice@email.com|555-0101|  Boston|
|        102|  Bob Jones|  bob@email.com|999-9999| Chicago|
|        104|  David Lee|david@email.com|555-0404|  Austin|
+-----------+-----------+---------------+--------+--------+



In [30]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, path)

In [31]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- location: string (nullable = true)



In [32]:
inc_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- location: string (nullable = true)



In [33]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, path)

In [34]:
deltaTable.alias("target").merge(
    inc_df.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(
    set={
        "name": "source.name",
        "email": "source.email",
        "phone": "source.phone",
        "location": "source.location"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "email": "source.email",
        "phone": "source.phone",
        "location": "source.location"
    }
).execute()

In [35]:
final_df = spark.read.format("delta").load(path)

final_df.show()

+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|  Boston|
|        102|    Bob Jones|    bob@email.com|999-9999| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        104|    David Lee|  david@email.com|555-0404|  Austin|
+-----------+-------------+-----------------+--------+--------+



In [36]:
print("Total Rows:", final_df.count())

Total Rows: 4


In [37]:
final_df.groupBy("customer_id") \
.count() \
.filter("count > 1") \
.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [38]:
final_df.orderBy("customer_id").show()

+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|  Boston|
|        102|    Bob Jones|    bob@email.com|999-9999| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        104|    David Lee|  david@email.com|555-0404|  Austin|
+-----------+-------------+-----------------+--------+--------+

